# COVID-19 Data Cleaning Project
**Dataset:** Our World in Data - COVID-19 Dataset  
**Tool:** Python (pandas)  
**Goal:** Clean the raw dataset and prepare it for analysis

## Step 1 - Loading the Data
Loading the CSV file and taking a first look at its size and structure.

In [ ]:
import pandas as pd
df= pd.read_csv('owid-covid-data.csv')
print (df.shape) 
print (df.head())
print (df.info())
print (df.describe())


(350085, 67)
  iso_code continent     location        date  total_cases  new_cases  \
0      AFG      Asia  Afghanistan  2020-01-03          NaN        0.0   
1      AFG      Asia  Afghanistan  2020-01-04          NaN        0.0   
2      AFG      Asia  Afghanistan  2020-01-05          NaN        0.0   
3      AFG      Asia  Afghanistan  2020-01-06          NaN        0.0   
4      AFG      Asia  Afghanistan  2020-01-07          NaN        0.0   

   new_cases_smoothed  total_deaths  new_deaths  new_deaths_smoothed  ...  \
0                 NaN           NaN         0.0                  NaN  ...   
1                 NaN           NaN         0.0                  NaN  ...   
2                 NaN           NaN         0.0                  NaN  ...   
3                 NaN           NaN         0.0                  NaN  ...   
4                 NaN           NaN         0.0                  NaN  ...   

   male_smokers  handwashing_facilities  hospital_beds_per_thousand  \
0           Na

## Step 2 - Auditing Missing Values
Counting how many nulls exist in each column and what percentage of the data is missing.

In [2]:
# Counting missing values per column
missing_count = df.isnull().sum()

# Calculating what percentage of each column is missing
missing_percent = (missing_count / len(df)) * 100

# Combining both into a neat table
missing_table = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %': missing_percent.round(2)
})

# Keep only columns that have missing values
missing_table = missing_table[missing_table['Missing Count'] > 0]

# Sort from worst to best and display
missing_table.sort_values('Missing %', ascending=False)


,Missing Count,Missing %
weekly_icu_admissions,339880,97.08
weekly_icu_admissions_per_million,339880,97.08
excess_mortality_cumulative_per_million,337901,96.52
excess_mortality_cumulative_absolute,337901,96.52
excess_mortality_cumulative,337901,96.52
...,...,...
new_deaths_smoothed_per_million,10804,3.09
new_cases_per_million,9628,2.75
new_cases,9628,2.75
new_deaths,9574,2.73


## Step 3 - Dropping Columns with Too Many Missing Values
Any column missing more than 70% of its data is not useful for analysis. We drop those columns entirely.

In [3]:
# Saving column count before dropping so we can compare later
before_cols= df.shape[1]
print("The column count before dropping", before_cols)

# Setting our threshold - drop any column that is more than 70% empty
threshold = 0.70

# Calculating minimum non-null values a column needs to survive
min_required = int((1-threshold)*len (df))

# Dropping columns that don't meet the minimum
df = df.dropna(thresh= min_required, axis=1) 

# Saving column count after dropping
after_cols= df.shape[1]
print ("Columns after dropping", after_cols)
print ("Columns removed", before_cols-after_cols )


The column count before dropping 67
Columns after dropping 38
Columns removed 29


## Step 4 - Separating Countries from Aggregate Regions
The dataset mixes real countries (India, USA) with regional aggregates (Asia, Africa, High income).
We separate them so our analysis only uses actual countries.

In [4]:
# Rows WITH a continent value = real countries
df_countries = df[df['continent'].notna()].copy()

# Rows WITHOUT a continent value = aggregates like Asia, Africa etc.
df_aggregates = df[df['continent'].isna()].copy()

print("Real country rows:", len(df_countries))
print("Aggregate rows:", len(df_aggregates))
print("Unique countries:", df_countries['location'].nunique())

Real country rows: 333420
Aggregate rows: 16665
Unique countries: 243


## Step 5 - Fixing the Date Column
The date column is currently stored as plain text.
We convert it to a proper datetime type so pandas can understand it as a date.

In [5]:
# Checking the current data type of date before fixing
print("Before:", df_countries['date'].dtype)

# Converting from text to proper datetime
df_countries['date'] = pd.to_datetime(df_countries['date'])

# Confirming it worked
print("After:", df_countries['date'].dtype)

# Checking the full date range of the dataset
print("Date range:", df_countries['date'].min(), "to", df_countries['date'].max())

Before: object
After: datetime64[ns]
Date range: 2020-01-01 00:00:00 to 2023-10-24 00:00:00


## Step 6 - Fixing Numeric Data Types
Key columns like total_cases and population are stored as decimals (float).
Since you can't have half a case or half a person, we convert them to whole numbers (int).

In [6]:
# Columns that should be whole numbers not decimals
int_cols = ['total_cases', 'new_cases', 'total_deaths', 'new_deaths', 'population']

# Looping through each column and fixing it
for col in int_cols:
    if col in df_countries.columns:
        # Must fill nulls with 0 first - pandas cannot convert NaN to int
        df_countries[col] = df_countries[col].fillna(0).astype(int)
        print(f"Fixed: {col}")

Fixed: total_cases
Fixed: new_cases
Fixed: total_deaths
Fixed: new_deaths
Fixed: population


## Step 7 - Filling Remaining Missing Values
For case and death columns, a missing value most likely means zero were reported that day.
We fill those with 0.

In [7]:
# Columns where missing likely means zero reported that day
fill_zero_cols = ['new_cases', 'new_deaths', 'new_cases_smoothed', 'new_deaths_smoothed']

for col in fill_zero_cols:
    if col in df_countries.columns:
        df_countries[col] = df_countries[col].fillna(0)
        print(f"Filled nulls in: {col}")

Filled nulls in: new_cases
Filled nulls in: new_deaths
Filled nulls in: new_cases_smoothed
Filled nulls in: new_deaths_smoothed


## Step 8 - Standardizing Text Columns
Removing any extra spaces from text columns and making sure they look consistent.

In [8]:
# Removing extra spaces from location and continent columns
df_countries['location'] = df_countries['location'].str.strip()
df_countries['continent'] = df_countries['continent'].str.strip()

# Checking unique continents to confirm they look clean
print("Continents:", df_countries['continent'].unique())

Continents: ['Asia' 'Europe' 'Africa' 'Oceania' 'North America' 'South America']


## Step 9 - Final Check
Reviewing the cleaned dataset - shape, remaining nulls, and a final preview.

In [9]:
print("=== FINAL CLEANED DATASET ===")
print("Shape:", df_countries.shape)

print("\nRemaining nulls (top 10):")
print(df_countries.isnull().sum().sort_values(ascending=False).head(10))

print("\nDate range:", df_countries['date'].min(), "to", df_countries['date'].max())
print("Countries covered:", df_countries['location'].nunique())

df_countries.head()

=== FINAL CLEANED DATASET ===
Shape: (333420, 38)

Remaining nulls (top 10):
tests_units                                   226632
handwashing_facilities                        201837
new_people_vaccinated_smoothed_per_hundred    165075
new_people_vaccinated_smoothed                165075
new_vaccinations_smoothed_per_million         164846
new_vaccinations_smoothed                     164846
extreme_poverty                               160249
reproduction_rate                             149679
stringency_index                              135769
male_smokers                                  133921
dtype: int64

Date range: 2020-01-01 00:00:00 to 2023-10-24 00:00:00
Countries covered: 243


,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,extreme_poverty,cardiovasc_death_rate,diabetes_prevalence,female_smokers,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index,population
0,AFG,Asia,Afghanistan,2020-01-03,0,0,0.0,0,0,0.0,...,NaN,597.029,9.59,NaN,NaN,37.746,0.5,64.83,0.511,41128772
1,AFG,Asia,Afghanistan,2020-01-04,0,0,0.0,0,0,0.0,...,NaN,597.029,9.59,NaN,NaN,37.746,0.5,64.83,0.511,41128772
2,AFG,Asia,Afghanistan,2020-01-05,0,0,0.0,0,0,0.0,...,NaN,597.029,9.59,NaN,NaN,37.746,0.5,64.83,0.511,41128772
3,AFG,Asia,Afghanistan,2020-01-06,0,0,0.0,0,0,0.0,...,NaN,597.029,9.59,NaN,NaN,37.746,0.5,64.83,0.511,41128772
4,AFG,Asia,Afghanistan,2020-01-07,0,0,0.0,0,0,0.0,...,NaN,597.029,9.59,NaN,NaN,37.746,0.5,64.83,0.511,41128772


## Step 10 - Saving the Cleaned Data
Exporting the final cleaned dataset as a new CSV file.
Always save to a new file - never overwrite your original raw data!

In [10]:
# Saving to a new file - never overwrite the original!
df_countries.to_csv('owid-covid-cleaned.csv', index=False)

print("Clean file saved successfully!")
print("Final shape:", df_countries.shape)

Clean file saved successfully!
Final shape: (333420, 38)


## Conclusion

### Problems Found in Raw Data:
- Several columns had 70%+ missing values — dropped
- Date column was stored as plain text — converted to datetime
- Dataset mixed real countries with regional aggregates — separated
- Numeric columns stored as floats — converted to integers
- Extra spaces in text columns — removed

### Result:
- Started with 67 columns — reduced to clean usable columns
- Dataset is now clean and ready for analysis